# Imports and setup

In [1]:
# Necessary imports and setup
import sys
import os

# Add execution tracking to debug duplicate output
print("=== STARTING IMPORTS CELL ===")

sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '../..')))

import multiprocessing as mp
try:
    mp.set_start_method('spawn', force=True)
except RuntimeError:
    pass

os.environ['CUDA_VISIBLE_DEVICES'] = '1'

# Configure JAX GPU memory settings BEFORE importing jax - OPTIMIZED FOR 40GB A100
os.environ['XLA_PYTHON_CLIENT_MEM_FRACTION'] = '0.99'  # Use 98% of GPU memory (~39.2GB out of 40GB)
os.environ['XLA_PYTHON_CLIENT_ALLOCATOR'] = 'cuda_async'  # Use platform allocator
os.environ['XLA_PYTHON_CLIENT_PREALLOCATE'] = 'false'  # Don't preallocate - grow as needed to avoid fragmentation
os.environ['TF_FORCE_GPU_ALLOW_GROWTH'] = 'true'  # Allow dynamic growth

# Tell XLA to use Triton GEMM, this improves steps/sec by ~30% on some GPUs
xla_flags = os.environ.get('XLA_FLAGS', '')
xla_flags += ' --xla_gpu_triton_gemm_any=True'
os.environ['XLA_FLAGS'] = xla_flags

import jax
from jax import numpy as jp
from jax.lib import xla_bridge

print("Device count: ", jax.device_count())

# Configure JAX to use only GPU1

print(f"CUDA_VISIBLE_DEVICES set to: {os.environ.get('CUDA_VISIBLE_DEVICES')}")
print(f"JAX memory fraction set to: {os.environ.get('XLA_PYTHON_CLIENT_MEM_FRACTION')}")
print(f"JAX preallocate disabled: {os.environ.get('XLA_PYTHON_CLIENT_PREALLOCATE')}")

print("JAX backend info:")
print(f"Platform: {xla_bridge.get_backend().platform}")
print(f"Device count: {xla_bridge.get_backend().device_count()}")
print(f"Devices: {xla_bridge.get_backend().devices()}")

# JAX configuration optimized for large workloads
jax.config.update('jax_enable_x64', False)  # Use float32 to save memory
jax.config.update('jax_traceback_filtering', 'off')
# Use bfloat16 for even better memory efficiency (optional - comment out if you need float32 precision)
# jax.config.update('jax_default_matmul_precision', 'bfloat16')

# Check GPU availability and memory
gpu_available = jax.devices()[0].platform == 'gpu'
print(f"GPU available: {gpu_available}")

if gpu_available:
    gpu_device = jax.devices('gpu')[0]
    print(f"GPU device: {gpu_device}")
else:
    print("No GPU device found.")

import signal
import json
import functools
import mujoco
from datetime import datetime
from pathlib import Path
import imageio
import gc

print("Basic imports completed...")

# Brax and training imports
from brax.io import model
from brax.training.agents.ppo import networks as ppo_networks
from brax.training.agents.ppo import train as ppo
from flax.training import orbax_utils
from orbax import checkpoint as ocp
from mujoco_playground.config import locomotion_params
from mujoco_playground import wrapper
from tensorboardX import SummaryWriter

print("Brax imports completed...")

# Task-specific imports
from tasks.common.randomize import domain_randomize as reachbot_randomize
from utils.telegram_messenger import send_message_sync

print("Task-specific imports completed...")

# Global variables
ENV_STR = 'Go1JoystickFlatTerrain'
x_data, y_data, y_dataerr = [], [], []
times = [datetime.now()]

# Signal handler for graceful interruption
def signal_handler(sig, frame):
    print('Program exited via keyboard interrupt')
    sys.exit(0)

signal.signal(signal.SIGINT, signal_handler)

from tasks.joystick.joystick import Joystick as ReachbotJoystick
from tasks.common.randomize import domain_randomize as reachbot_randomize
from mujoco_playground import registry


# JSON encoder for JAX arrays
class JaxArrayEncoder(json.JSONEncoder):
    def default(self, obj):
        if isinstance(obj, jp.ndarray):
            return obj.tolist()
        return json.JSONEncoder.default(self, obj)

print("=== ALL IMPORTS LOADED SUCCESSFULLY! ===")

=== STARTING IMPORTS CELL ===
Device count:  1
CUDA_VISIBLE_DEVICES set to: 1
JAX memory fraction set to: 0.99
JAX preallocate disabled: false
JAX backend info:
Platform: gpu
Device count: 1
Devices: [CudaDevice(id=0)]
GPU available: True
GPU device: cuda:0
Basic imports completed...


/tmp/ipykernel_2596361/1701478872.py:42: DeprecationWarning: jax.lib.xla_bridge.get_backend is deprecated; use jax.extend.backend.get_backend.
  print(f"Platform: {xla_bridge.get_backend().platform}")


Brax imports completed...
Task-specific imports completed...
=== ALL IMPORTS LOADED SUCCESSFULLY! ===


## 🔌 SSH Disconnection Recovery Guide

If your SSH connection gets interrupted, here's how to check if your training is still running and monitor progress:

### 1. Check if the process is still running:
```bash
ps aux | grep python | grep cave_exploration
# or
ps aux | grep jupyter
```

### 2. Monitor the training logs:
```bash
# Check TensorBoard logs
ls -la logs/cave_exploration-*/
tail -f logs/cave_exploration-*/events.out.tfevents.*

# Monitor any output files
tail -f notebook_output_*.log
```

### 3. Check GPU usage (confirms training is active):
```bash
nvidia-smi
# or watch it continuously
watch -n 1 nvidia-smi
```

### 4. Kill the process if needed:
```bash
# Find the PID
ps aux | grep python | grep cave_exploration
# Kill gracefully
kill -TERM <PID>
# Force kill if needed
kill -9 <PID>
```

### 5. Alternative: Use tmux/screen for future sessions:
```bash
# Start a tmux session before running notebook
tmux new-session -d -s training
tmux attach -t training
# Then run your notebook - it will survive SSH disconnections
```

**The notebook is now configured to ignore SSH disconnections and continue training!** 🚀

In [2]:
# SSH Disconnection Resilience - Keep Running Independent of SSH Session
import signal
import os

print("=== SETTING UP SSH DISCONNECTION RESILIENCE ===")

def ignore_sighup(signum, frame):
    """Ignore SIGHUP signal (SSH disconnection) to keep process running"""
    print(f"\n🔌 SSH disconnection detected (SIGHUP), but continuing to run in background...")
    print(f"📋 Process PID: {os.getpid()}")
    print("🚀 Training will continue independently!")

def handle_sigterm(signum, frame):
    """Handle SIGTERM gracefully but continue for most cases"""
    print(f"\n⚠️  Received SIGTERM, but attempting to continue...")
    print("🔄 If this is a system shutdown, the process will be force-killed anyway")

def handle_sigint(signum, frame):
    """Handle Ctrl+C - this one we do want to respect for manual interruption"""
    print(f"\n🛑 Received SIGINT (Ctrl+C) - User requested interruption")
    print("💾 Cleaning up gracefully...")
    exit(0)

# Set up signal handlers
signal.signal(signal.SIGHUP, ignore_sighup)    # SSH disconnection - IGNORE
signal.signal(signal.SIGTERM, handle_sigterm)  # Termination - TRY TO IGNORE  
signal.signal(signal.SIGINT, handle_sigint)    # Ctrl+C - RESPECT

# Additional resilience measures
print("🛡️  SSH Disconnection Resilience Active!")
print(f"📋 Process PID: {os.getpid()}")
print("🔌 SSH disconnections will be ignored")
print("🚀 Notebook will continue running in background")
print("⚠️  Note: Only Ctrl+C will stop the training")

# Optional: Redirect stdout/stderr to files for monitoring after SSH disconnect
import sys
from datetime import datetime

# Create a log file for output monitoring
log_file = f"notebook_output_{datetime.now().strftime('%Y%m%d_%H%M%S')}.log"
print(f"📝 Consider tailing this log file after SSH reconnection: {log_file}")

# Function to setup output redirection (optional - uncomment if needed)
def redirect_output_to_file():
    """Redirect stdout and stderr to a file - useful for monitoring after disconnect"""
    log_path = os.path.join(os.getcwd(), log_file)
    sys.stdout = open(log_path, 'a')
    sys.stderr = sys.stdout
    print(f"Output redirected to: {log_path}")

# Uncomment the next line if you want output redirected to a file
# redirect_output_to_file()

print("✅ SSH resilience setup complete!")

=== SETTING UP SSH DISCONNECTION RESILIENCE ===
🛡️  SSH Disconnection Resilience Active!
📋 Process PID: 2596361
🔌 SSH disconnections will be ignored
🚀 Notebook will continue running in background
⚠️  Note: Only Ctrl+C will stop the training
📝 Consider tailing this log file after SSH reconnection: notebook_output_20251009_225452.log
✅ SSH resilience setup complete!


# Environment config

In [3]:
from tasks.joystick.joystick import default_config as reachbot_config
env_cfg = reachbot_config()

# Basic simulation parameters
env_cfg.sim_dt = 0.004
env_cfg.action_scale = 1

# PID control parameters
env_cfg.Kp_pri = 60.0
env_cfg.Kd_pri = 20.0
env_cfg.Kp_rot = 25.0
env_cfg.Kd_rot = 2

env_cfg.noise_config.level = 0.0
#env_cfg.reward_config.scales.posture = 0.0
#env_cfg.reward_config.dof_vel = -0.1
env_cfg.reward_config.scales.feet_clearance=0
env_cfg.reward_config.scales.feet_height=-1
env_cfg.reward_config.scales.feet_slip=-0.1
env_cfg.reward_config.scales.feet_air_time=0
env_cfg.reward_config.scales.correct_height=-0.0  
#env_cfg.reward_config.scales.torques=0#-0.0002,            # Penalty scale for torques applied. Default is -0.0002
#env_cfg.reward_config.scales.action_rate=0#-0.01,          # Penalty scale for action rate changes. Default is -0.01
#env_cfg.reward_config.scales.energy=-0.001,      
env_cfg.reward_config.scales.orientation=-0.5#-0.2,          # Penalty scale for orientation. Default is -0.1
env_cfg.pert_config.enable = True


print("Environment configuration completed!")


Environment configuration completed!


# Training parameters

In [4]:
ppo_params = locomotion_params.brax_ppo_config(ENV_STR)
ppo_training_params = dict(ppo_params)

ppo_training_params["num_timesteps"] = 50_000_000

# Training


In [ ]:
import os
import threading
print(f"🔥 PID: {os.getpid()} | Thread: {threading.current_thread().ident} | Time: {datetime.now()}")

# Training execution
# Create log directory for training run
datetime_str = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
main_dir = os.path.dirname(os.path.dirname(os.getcwd()))
logdir = os.path.join(main_dir, "logs", "joystick-"+datetime_str)
os.makedirs(logdir, exist_ok=True)

# Create environment
env = ReachbotJoystick(config=env_cfg)

# Initialize tracking variables
timesteps = []
rewards = []
total_rewards = []
total_rewards_std = []
times = [datetime.now()]

writer = SummaryWriter(logdir=logdir)

# Progress tracking function
def progress(num_steps, metrics):
    """Function to track progress and log metrics during training."""
    print(f"Progress at step {num_steps}: {metrics}")
    # Log to TensorBoard
    for key, value in metrics.items():
        if not (jp.isnan(value) or jp.isinf(value)):
            writer.add_scalar(key, value, num_steps)
        else:
            print(f"Warning: Skipping NaN/Inf value for metric '{key}' at step {num_steps}")

    if "eval/episode_reward" in metrics:
        episode_reward = metrics["eval/episode_reward"]
        if jp.isnan(episode_reward) or jp.isinf(episode_reward):
            print("Warning: NaN/Inf reward encountered, aborting.")
            run_duration = str(datetime.now() - times[0])
            send_message_sync(
                task="Joystick RL Training",
                duration=run_duration,
                result="Failed: NaN/Inf reward encountered"
            )
            raise ValueError(f"NaN/Inf reward encountered at step {num_steps}: {episode_reward}")
        
        times.append(datetime.now())
        timesteps.append(num_steps)
        total_rewards.append(episode_reward)
        total_rewards_std.append(metrics["eval/episode_reward_std"])
    
        writer.flush()
        metrics["timesteps"] = num_steps
        metrics["time"] = (times[-1] - times[0]).total_seconds()
        rewards.append(metrics)
        
        percent_complete = (num_steps / ppo_training_params["num_timesteps"]) * 100
        if num_steps == 0:
            remaining_time_str = "unknown"
        else:
            remaining_time = (ppo_training_params["num_timesteps"] - num_steps) * (times[-1] - times[0]).total_seconds() / num_steps / 60
            remaining_time_str = f"{remaining_time:.2f}"
        
        print(f"step: {num_steps}/{ppo_training_params['num_timesteps']} ({percent_complete:.1f}%), reward: {total_rewards[-1]:.3f} +/- {total_rewards_std[-1]:.3f}, time passed (min): {(times[-1] - times[0]).total_seconds() / 60:.2f} min, calculated time left (min): {remaining_time_str} min")

# Network factory setup
network_factory = ppo_networks.make_ppo_networks(observation_size=env.observation_size, action_size=env.action_size)
if "network_factory" in ppo_params:
    if "network_factory" in ppo_training_params:
        del ppo_training_params["network_factory"]
    network_factory = functools.partial(
        ppo_networks.make_ppo_networks,
        **ppo_params.network_factory
    )
print("Created neural network with input size:", env.observation_size, "and output size:", env.action_size)

# Checkpoint saving function
def policy_params_fn(current_step, make_policy, params):
    del make_policy  # Unused.
    orbax_checkpointer = ocp.PyTreeCheckpointer()
    save_args = orbax_utils.save_args_from_target(params)
    checkpoint_path = os.path.join(logdir, 'checkpoints')
    path = os.path.join(checkpoint_path, f"{current_step}")
    abs_path = os.path.abspath(path)
    orbax_checkpointer.save(abs_path, params, force=True, save_args=save_args)

# Save configurations
print("Saving configs")
configs = {
    "env_cfg": env_cfg.to_dict(),
    "ppo_params": ppo_training_params,
}

def replace_infinity(obj):
    if isinstance(obj, dict):
        return {k: replace_infinity(v) for k, v in obj.items()}
    elif isinstance(obj, list):
        return [replace_infinity(v) for v in obj]
    elif isinstance(obj, float) and obj == float('inf'):
        return 1e308
    return obj

configs = replace_infinity(configs)
config_path = os.path.join(logdir, 'config.json')
with open(config_path, "w", encoding="utf-8") as fp:
    json.dump(configs, fp, indent=4)
print(f"Configuration saved to {config_path}")
writer.add_text('config', json.dumps(configs, indent=4))

# Setup training function
randomizer = reachbot_randomize
train_fn = functools.partial(
    ppo.train, 
    **dict(ppo_training_params),
    network_factory=network_factory,
    progress_fn=progress,
    policy_params_fn=policy_params_fn,
    max_devices_per_host=1,
    log_training_metrics=True,
)

# Run training
print("Training the model...")
try:
    make_inference_fn, params, metrics = train_fn(
        environment=env,
        wrap_env_fn=wrapper.wrap_for_brax_training,
    )
    print("Training completed successfully!")
except Exception as e:
    import traceback
    run_duration = str(datetime.now() - times[0])
    send_message_sync(
        task="Joystick RL Training",
        duration=run_duration,
        result=f"Failed: {e}"
    )
    traceback.print_exc()
    raise

print(f"time to jit: {times[1] - times[0]}")
print(f"time to train: {times[-1] - times[1]}")

# Save results
results_path = os.path.join(logdir, 'results.txt')
with open(results_path, 'w') as f:
    for i in range(len(total_rewards)):
        f.write(f"step: {timesteps[i]}, reward: {total_rewards[i]}, reward_std: {total_rewards_std[i]}\n")
    f.write(f"Time to jit: {times[1] - times[0]}\n")
    f.write(f"Time to train: {times[-1] - times[1]}\n")

# Save rewards as JSON
def nest_flat_dict(flat_dict):
    nested_dict = {}
    for key, value in flat_dict.items():
        parts = key.split('/')
        d = nested_dict
        for i, part in enumerate(parts):
            is_last_part = (i == len(parts) - 1)
            if is_last_part:
                if isinstance(d.get(part), dict):
                    d[part]['value'] = value
                else:
                    d[part] = value
            else:
                if not isinstance(d.get(part), dict):
                    d[part] = {'value': d[part]} if part in d else {}
                d = d[part]
    return nested_dict

nested_rewards = [nest_flat_dict(r) for r in rewards]
rewards_path = os.path.join(logdir, 'rewards.json')
with open(rewards_path, 'w') as fp:
    json.dump(nested_rewards, fp, indent=4, cls=JaxArrayEncoder)

# Save trained parameters
params_path = os.path.join(logdir, 'params')
model.save_params(params_path, params)

print(f"Training completed! Results saved to: {logdir}")
print(f"Final reward: {total_rewards[-1]:.3f} ± {total_rewards_std[-1]:.3f}")

# Store these variables for the video generation cell
trained_params = params
trained_make_inference_fn = make_inference_fn
trained_env = env
trained_logdir = logdir
# Video creation from trained model
# Free up training memory before rendering
del train_fn, network_factory, writer
gc.collect()

# Use the already loaded model and environment from the training cell
env = trained_env
params = trained_params
make_inference_fn = trained_make_inference_fn
logdir = trained_logdir

# Setup JIT compiled functions for inference
jit_reset = jax.jit(env.reset)
jit_step = jax.jit(env.step)
inference_fn = make_inference_fn(params, deterministic=True)
jit_inference_fn = jax.jit(inference_fn)

print("Setting up rollout for video creation...")

# Rollout parameters
rng = jax.random.PRNGKey(0)
rollout = []
n_episodes = 1
rollout_steps = 2000

# Set command (if needed for environment)
x_vel = 0.2
y_vel = 0.2
yaw_vel = 0.0
command = jp.array([x_vel, y_vel, yaw_vel])

# Rollout policy and record simulation
print(f"Running rollout for {n_episodes} episode(s) with {rollout_steps} steps each...")
for episode in range(n_episodes):
    print(f"Episode {episode + 1}/{n_episodes}")
    state = jit_reset(rng)
    rollout.append(state)
    episode_reward = 0.0
    
    for i in range(rollout_steps):
        if i % 500 == 0:
            print(f"  Step {i}/{rollout_steps} - Total Reward: {episode_reward:.3f}")
            
        act_rng, rng = jax.random.split(rng)
        ctrl, _ = jit_inference_fn(state.obs, act_rng)
        
        # Check for numerical issues
        if jp.any(jp.isinf(ctrl)) or jp.any(jp.isnan(ctrl)):
            print(f"Numerical issue detected in control at step {i}. Stopping rollout.")
            break
            
        state = jit_step(state, ctrl)
        
        # Set command if the environment supports it
        if hasattr(state, 'info') and 'command' in state.info:
            state.info["command"] = command

        episode_reward += state.reward
            
        rollout.append(state)

    print(f"Rollout completed with {len(rollout)} states")

    # Render video
    print("Rendering video...")
    render_every = 1  # Render every frame
    width = 1920      # Full HD width
    height = 1080     # Full HD height

    frames = env.render(rollout[::render_every], camera='track', width=width, height=height)
    print(f"Rendered {len(frames)} frames")

    # Save video
    video_path = os.path.join(logdir, f'posttraining_{episode_reward:.2f}.mp4')
    fps = 1.0 / env.dt

    print(f"Saving video to {video_path} at {fps} FPS...")
    imageio.mimsave(video_path, frames, fps=fps)
    print(f"Video saved successfully to {video_path}")

# Send completion notification
run_duration = str(times[-1] - times[0])
if total_rewards:
    result = f"Final reward: {total_rewards[-1]:.3f} ± {total_rewards_std[-1]:.3f}"
else:
    result = "No rewards recorded."

send_message_sync(
    task="Joystick RL Training",
    duration=run_duration,
    result=result
)

print("\n=== TRAINING AND VIDEO CREATION COMPLETE ===")
print(f"Log directory: {logdir}")
print(f"Video file: {video_path}")
print(f"Training duration: {run_duration}")
print(f"Final result: {result}")

🔥 PID: 2596361 | Thread: 139793086001792 | Time: 2025-10-09 22:54:52.721172


Precomputed 60 LIDAR ray directions
CaveExplore task action space: 12
Created neural network with input size: {'privileged_state': (183,), 'state': (108,)} and output size: 12
Saving configs
Configuration saved to /home/ga53voq/master_thesis/tasks/joystick/logs/joystick-2025-10-09_22-54-52/config.json
Training the model...


2025-10-09 22:57:10.327015: W external/xla/xla/service/gpu/autotuning/dot_search_space.cc:200] All configs were filtered out because none of them sufficiently match the hints. Maybe the hints set does not contain a good representative set of valid configs?Working around this by using the full hints set instead.
2025-10-09 22:57:10.327152: W external/xla/xla/service/gpu/autotuning/dot_search_space.cc:200] All configs were filtered out because none of them sufficiently match the hints. Maybe the hints set does not contain a good representative set of valid configs?Working around this by using the full hints set instead.
2025-10-09 22:57:10.327166: W external/xla/xla/service/gpu/autotuning/dot_search_space.cc:200] All configs were filtered out because none of them sufficiently match the hints. Maybe the hints set does not contain a good representative set of valid configs?Working around this by using the full hints set instead.


Progress at step 0: {'eval/walltime': 125.74322319030762, 'eval/episode_reward': Array(2.875311, dtype=float32), 'eval/episode_reward/action_rate': Array(-29.43351, dtype=float32), 'eval/episode_reward/ang_vel_xy': Array(-132.55782, dtype=float32), 'eval/episode_reward/boom_extension_speed': Array(-22.07991, dtype=float32), 'eval/episode_reward/correct_height': Array(0., dtype=float32), 'eval/episode_reward/dof_pos_limits': Array(-0.03080971, dtype=float32), 'eval/episode_reward/energy': Array(-170.62051, dtype=float32), 'eval/episode_reward/feet_air_time': Array(0., dtype=float32), 'eval/episode_reward/feet_clearance': Array(0., dtype=float32), 'eval/episode_reward/feet_height': Array(-96.82086, dtype=float32), 'eval/episode_reward/feet_slip': Array(-1.6903551, dtype=float32), 'eval/episode_reward/lin_vel_z': Array(-11.016571, dtype=float32), 'eval/episode_reward/orientation': Array(-30.377611, dtype=float32), 'eval/episode_reward/pose': Array(0., dtype=float32), 'eval/episode_reward/

2025-10-09 23:09:06.687581: W external/xla/xla/service/gpu/autotuning/dot_search_space.cc:200] All configs were filtered out because none of them sufficiently match the hints. Maybe the hints set does not contain a good representative set of valid configs?Working around this by using the full hints set instead.


  Step 0/2000 - Total Reward: 0.000
  Step 500/2000 - Total Reward: 27.013
  Step 1000/2000 - Total Reward: 55.435
  Step 1500/2000 - Total Reward: 83.813
Rollout completed with 2001 states
Rendering video...


100%|██████████| 2001/2001 [00:16<00:00, 124.97it/s]
/home/ga53voq/.conda/envs/pyenv/lib/python3.12/subprocess.py:1885: RuntimeWarning: os.fork() was called. os.fork() is incompatible with multithreaded code, and JAX is multithreaded, so this will likely lead to a deadlock.
  self.pid = _fork_exec(


Rendered 2001 frames
Saving video to /home/ga53voq/master_thesis/tasks/joystick/logs/joystick-2025-10-09_22-54-52/posttraining_112.11.mp4 at 50.0 FPS...
Video saved successfully to /home/ga53voq/master_thesis/tasks/joystick/logs/joystick-2025-10-09_22-54-52/posttraining_112.11.mp4

=== TRAINING AND VIDEO CREATION COMPLETE ===
Log directory: /home/ga53voq/master_thesis/tasks/joystick/logs/joystick-2025-10-09_22-54-52
Video file: /home/ga53voq/master_thesis/tasks/joystick/logs/joystick-2025-10-09_22-54-52/posttraining_112.11.mp4
Training duration: 0:13:54.609496
Final result: Final reward: 49.731 ± 8.505


# Output video render

In [6]:
# Video creation from trained model


# Use the already loaded model and environment from the training cell
env = trained_env
params = trained_params
make_inference_fn = trained_make_inference_fn
logdir = trained_logdir

# Setup JIT compiled functions for inference
jit_reset = jax.jit(env.reset)
jit_step = jax.jit(env.step)
inference_fn = make_inference_fn(params, deterministic=False)
jit_inference_fn = jax.jit(inference_fn)

print("Setting up rollout for video creation...")

# Rollout parameters
rng = jax.random.PRNGKey(0)
rollout = []
n_episodes = 5
rollout_steps = 10000

# Set command (if needed for environment)
x_vel = 0.2
y_vel = 0.2
yaw_vel = 0.0
command = jp.array([x_vel, y_vel, yaw_vel])

# Rollout policy and record simulation
print(f"Running rollout for {n_episodes} episode(s) with {rollout_steps} steps each...")
episode_rewards = []

for episode in range(n_episodes):
    print(f"Episode {episode + 1}/{n_episodes}")
    state = jit_reset(rng)
    rollout = [state]  # Reset rollout for each episode
    episode_reward = 0.0
    
    for i in range(rollout_steps):
        if i % 500 == 0:
            print(f"  Step {i}/{rollout_steps}, Current reward: {episode_reward:.3f}")
            
        act_rng, rng = jax.random.split(rng)
        ctrl, _ = jit_inference_fn(state.obs, act_rng)
        
        # Check for numerical issues
        if jp.any(jp.isinf(ctrl)) or jp.any(jp.isnan(ctrl)):
            print(f"Numerical issue detected in control at step {i}. Stopping rollout.")
            break
            
        state = jit_step(state, ctrl)
        
        # Accumulate reward for this episode
        episode_reward += float(state.reward)
        
        if state.done:
            print(f"Episode {episode + 1} ended at step {i} with reward: {episode_reward:.3f}")
            break
            
        rollout.append(state)

    episode_rewards.append(episode_reward)
    print(f"Episode {episode + 1} completed with {len(rollout)} states and total reward: {episode_reward:.3f}")

    # Render video
    print("Rendering video...")
    render_every = 1  # Render every frame
    width = 1920      # Full HD width
    height = 1080     # Full HD height

    frames = env.render(rollout[::render_every], camera='track_global', width=width, height=height)
    print(f"Rendered {len(frames)} frames")

    # Save video
    video_path = os.path.join(logdir, f'posttraining_episode_{episode}_reward_{episode_reward:.1f}.mp4')
    fps = 1.0 / env.dt

    print(f"Saving video to {video_path} at {fps} FPS...")
    imageio.mimsave(video_path, frames, fps=fps)
    print(f"Video saved successfully: Episode {episode + 1}, Reward: {episode_reward:.3f}")

# Print summary of all episodes
print("\n=== EPISODE REWARD SUMMARY ===")
for i, reward in enumerate(episode_rewards):
    print(f"Episode {i + 1}: {reward:.3f}")
print(f"Average reward: {sum(episode_rewards)/len(episode_rewards):.3f}")
print(f"Best episode: {episode_rewards.index(max(episode_rewards)) + 1} with reward {max(episode_rewards):.3f}")
print(f"Worst episode: {episode_rewards.index(min(episode_rewards)) + 1} with reward {min(episode_rewards):.3f}")

# Send completion notification
run_duration = str(times[-1] - times[0])
if total_rewards:
    training_result = f"Training final reward: {total_rewards[-1]:.3f} ± {total_rewards_std[-1]:.3f}"
else:
    training_result = "No training rewards recorded."

# Include episode rewards in notification
episode_summary = f"Episode rewards: {[f'{r:.1f}' for r in episode_rewards]}, Avg: {sum(episode_rewards)/len(episode_rewards):.1f}"

send_message_sync(
    task="Joystick RL Training",
    duration=run_duration,
    result=f"{training_result}\n{episode_summary}"
)

print("\n=== TRAINING AND VIDEO CREATION COMPLETE ===")
print(f"Log directory: {logdir}")
print(f"Training duration: {run_duration}")
print(f"Training result: {training_result}")
print(f"Episode summary: {episode_summary}")

Setting up rollout for video creation...
Running rollout for 5 episode(s) with 10000 steps each...
Episode 1/5
  Step 0/10000, Current reward: 0.000
  Step 500/10000, Current reward: 25.442
  Step 1000/10000, Current reward: 52.401
  Step 1500/10000, Current reward: 78.183
  Step 2000/10000, Current reward: 105.193
  Step 2500/10000, Current reward: 132.094
  Step 3000/10000, Current reward: 159.495
  Step 3500/10000, Current reward: 186.451
  Step 4000/10000, Current reward: 213.438
  Step 4500/10000, Current reward: 240.347
  Step 5000/10000, Current reward: 266.877
  Step 5500/10000, Current reward: 293.108
  Step 6000/10000, Current reward: 319.545
  Step 6500/10000, Current reward: 345.964
  Step 7000/10000, Current reward: 372.441
  Step 7500/10000, Current reward: 399.649
  Step 8000/10000, Current reward: 426.751
  Step 8500/10000, Current reward: 453.871
  Step 9000/10000, Current reward: 480.932
  Step 9500/10000, Current reward: 507.852
Episode 1 completed with 10001 states 

100%|██████████| 10001/10001 [01:21<00:00, 123.05it/s]


Rendered 10001 frames
Saving video to /home/ga53voq/master_thesis/tasks/joystick/logs/joystick-2025-10-09_22-54-52/posttraining_episode_0_reward_534.9.mp4 at 50.0 FPS...
Video saved successfully: Episode 1, Reward: 534.948
Episode 2/5
  Step 0/10000, Current reward: 0.000
  Step 500/10000, Current reward: 22.000
  Step 1000/10000, Current reward: 48.519
  Step 1500/10000, Current reward: 75.588
  Step 2000/10000, Current reward: 98.156
  Step 2500/10000, Current reward: 124.771
  Step 3000/10000, Current reward: 151.690
  Step 3500/10000, Current reward: 173.396
  Step 4000/10000, Current reward: 200.017
  Step 4500/10000, Current reward: 227.106
  Step 5000/10000, Current reward: 254.417
  Step 5500/10000, Current reward: 281.651
  Step 6000/10000, Current reward: 309.045
  Step 6500/10000, Current reward: 335.764
  Step 7000/10000, Current reward: 361.667
  Step 7500/10000, Current reward: 387.675
  Step 8000/10000, Current reward: 414.407
  Step 8500/10000, Current reward: 441.453
 

100%|██████████| 10001/10001 [01:24<00:00, 118.65it/s]


Rendered 10001 frames
Saving video to /home/ga53voq/master_thesis/tasks/joystick/logs/joystick-2025-10-09_22-54-52/posttraining_episode_1_reward_519.4.mp4 at 50.0 FPS...
Video saved successfully: Episode 2, Reward: 519.356
Episode 3/5
  Step 0/10000, Current reward: 0.000
  Step 500/10000, Current reward: 25.288
  Step 1000/10000, Current reward: 50.787
  Step 1500/10000, Current reward: 76.537
  Step 2000/10000, Current reward: 101.501
  Step 2500/10000, Current reward: 127.779
  Step 3000/10000, Current reward: 154.081
  Step 3500/10000, Current reward: 180.241
  Step 4000/10000, Current reward: 206.163
  Step 4500/10000, Current reward: 230.621
  Step 5000/10000, Current reward: 244.013
Episode 3 ended at step 5462 with reward: 264.504
Episode 3 completed with 5463 states and total reward: 264.504
Rendering video...


...

Rendered 5463 frames
Saving video to /home/ga53voq/master_thesis/tasks/joystick/logs/joystick-2025-10-09_22-54-52/posttraining_episode_2_reward_264.5.mp4 at 50.0 FPS...
Video saved successfully: Episode 3, Reward: 264.504
Episode 4/5
  Step 0/10000, Current reward: 0.000
  Step 500/10000, Current reward: 25.502
  Step 1000/10000, Current reward: 50.080
  Step 1500/10000, Current reward: 74.968
  Step 2000/10000, Current reward: 99.502
Episode 4 ended at step 2171 with reward: 105.752
Episode 4 completed with 2172 states and total reward: 105.752
Rendering video...


100%|██████████| 2172/2172 [00:15<00:00, 137.55it/s]


Rendered 2172 frames
Saving video to /home/ga53voq/master_thesis/tasks/joystick/logs/joystick-2025-10-09_22-54-52/posttraining_episode_3_reward_105.8.mp4 at 50.0 FPS...
Video saved successfully: Episode 4, Reward: 105.752
Episode 5/5
  Step 0/10000, Current reward: 0.000
  Step 500/10000, Current reward: 26.241
  Step 1000/10000, Current reward: 52.913
  Step 1500/10000, Current reward: 79.343
  Step 2000/10000, Current reward: 105.497
  Step 2500/10000, Current reward: 131.755
Episode 5 ended at step 2764 with reward: 142.415
Episode 5 completed with 2765 states and total reward: 142.415
Rendering video...


100%|██████████| 2765/2765 [00:21<00:00, 128.88it/s]


Rendered 2765 frames
Saving video to /home/ga53voq/master_thesis/tasks/joystick/logs/joystick-2025-10-09_22-54-52/posttraining_episode_4_reward_142.4.mp4 at 50.0 FPS...
Video saved successfully: Episode 5, Reward: 142.415

=== EPISODE REWARD SUMMARY ===
Episode 1: 534.948
Episode 2: 519.356
Episode 3: 264.504
Episode 4: 105.752
Episode 5: 142.415
Average reward: 313.395
Best episode: 1 with reward 534.948
Worst episode: 4 with reward 105.752

=== TRAINING AND VIDEO CREATION COMPLETE ===
Log directory: /home/ga53voq/master_thesis/tasks/joystick/logs/joystick-2025-10-09_22-54-52
Training duration: 0:13:54.609496
Training result: Training final reward: 49.731 ± 8.505
Episode summary: Episode rewards: ['534.9', '519.4', '264.5', '105.8', '142.4'], Avg: 313.4
